In [1]:
from snowflake.snowpark.functions import col,trim,split,lit
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
import sys 
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np
import datetime
import math 
from dateutil.relativedelta import relativedelta
from snowflake_utils import *

### Snowflake Session

In [2]:
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

### Python UDF

In [3]:
def remove_quotes_existing_columns(df):
    for old_col in df.columns:
        new_col = old_col.replace('"','')
        df = df.rename(old_col,new_col)
    return df

### Configuration

In [15]:
TABLE_NAME = 'MOP_DATABASE.SOQ.DAILY_DATA_WITH_FESTIVE_FEATURES'

### Read the table

In [16]:
data = session.table(TABLE_NAME)

In [17]:
data.select(F.count_distinct("PARENT_DEALER_CODE_MODEL_FAMILY")).show()

------------------------------------------------------
|"COUNT( DISTINCT ""PARENT_DEALER_CODE_MODEL_FAM...  |
------------------------------------------------------
|104087                                              |
------------------------------------------------------



In [18]:
data.select(F.min("CAL_DATE").alias("EARLIEST_CAL_DATE")).show()

-----------------------
|"EARLIEST_CAL_DATE"  |
-----------------------
|2023-04-01           |
-----------------------



In [19]:
data.select(F.max("CAL_DATE").alias("LATEST_CAL_DATE")).show()

---------------------
|"LATEST_CAL_DATE"  |
---------------------
|2026-12-10         |
---------------------



In [20]:
TODAYS_DATE = datetime.date.today()

In [21]:
TODAYS_DATE

datetime.date(2026, 9, 18)

In [22]:
#Filter for today
TODAYS_DATE = datetime.date.today()
df = data.filter((F.col("CAL_DATE")<=TODAYS_DATE))

In [23]:
from snowflake.snowpark import functions as F
from snowflake.snowpark.window import Window

# 2. Aggregate sales to the Series + Month level
# (This sums up daily sales into monthly buckets per series)
df_monthly_sales = df.group_by("YEAR", "MONTH", "PARENT_DEALER_CODE_MODEL_FAMILY") \
                               .agg(F.sum("NET_SALES").alias("SERIES_MONTHLY_SALES"))

window_total = Window.partition_by("YEAR", "MONTH")
window_running = Window.partition_by("YEAR", "MONTH").order_by(F.desc("SERIES_MONTHLY_SALES"))

# 2. Build the base running percentages
df_base = df_monthly_sales.with_column(
    "TOTAL_MONTHLY_SALES", F.sum("SERIES_MONTHLY_SALES").over(window_total)
).with_column(
    "RUNNING_SUM_SALES", F.sum("SERIES_MONTHLY_SALES").over(window_running)
).with_column(
    "CUMULATIVE_SALES_PERCENTAGE", (F.col("RUNNING_SUM_SALES") / F.col("TOTAL_MONTHLY_SALES")) * 100
)

# 3. Look at the previous row's cumulative percentage to ensure we catch the border-crossing series
df_with_lag = df_base.with_column(
    "PREV_CUM_PERCENTAGE", F.lag("CUMULATIVE_SALES_PERCENTAGE", offset=1, default_value=0.0).over(window_running)
)

# 4. Label the series using an F.when() conditional statement
df_labeled = df_with_lag.with_column(
    "SALES_BENEFACTOR_85",
    F.when(F.col("PREV_CUM_PERCENTAGE") < 85.0, F.lit("Top 85% Driver"))
     .otherwise(F.lit("Long Tail / Other"))
)

In [24]:
verification_df = df_labeled.group_by("YEAR", "MONTH").agg(
    # Count how many total series exist in that month
    F.count("PARENT_DEALER_CODE_MODEL_FAMILY").alias("TOTAL_SERIES_COUNT"),
    
    # EMULATE count_if: Sum a 1 if true, 0 if false
    F.sum(F.when(F.col("SALES_BENEFACTOR_85") == "Top 85% Driver", 1).otherwise(0)).alias("TOP_SERIES_COUNT"),
    
    # Sum up the total sales of the month
    F.sum("SERIES_MONTHLY_SALES").alias("MONTH_TOTAL_SALES"),
    
    # EMULATE sum_if: Sum the column value if true, 0 if false
    F.sum(F.when(F.col("SALES_BENEFACTOR_85") == "Top 85% Driver", F.col("SERIES_MONTHLY_SALES")).otherwise(0)).alias("TOP_SERIES_SALES")
).with_column(
    # 2. Calculate the actual percentages to verify the 80/20 mix
    "PERCENT_OF_SERIES", (F.col("TOP_SERIES_COUNT") / F.col("TOTAL_SERIES_COUNT")) * 100
).with_column(
    "PERCENT_OF_SALES", (F.col("TOP_SERIES_SALES") / F.col("MONTH_TOTAL_SALES")) * 100
)

# 3. Select columns and display chronologically
verification_df.select(
    "YEAR", 
    "MONTH", 
    F.round("PERCENT_OF_SERIES", 1).alias("SERIES_PCT"), 
    F.round("PERCENT_OF_SALES", 1).alias("SALES_PCT")
).sort("YEAR", "MONTH").show()

-------------------------------------------------
|"YEAR"  |"MONTH"  |"SERIES_PCT"  |"SALES_PCT"  |
-------------------------------------------------
|2023    |4        |7.1           |84.6         |
|2023    |5        |6.7           |84.3         |
|2023    |6        |9.1           |84.7         |
|2023    |7        |8.9           |84.1         |
|2023    |8        |9.4           |84.0         |
|2023    |9        |8.6           |83.4         |
|2023    |10       |9.1           |83.7         |
|2023    |11       |7.8           |84.7         |
|2023    |12       |8.2           |82.7         |
|2024    |1        |8.5           |84.1         |
-------------------------------------------------



In [25]:
df_top_drivers = df_labeled.filter(F.col("SALES_BENEFACTOR_85") == "Top 85% Driver")

# 2. Group by the Series to see how consistently they show up
df_consistency = df_top_drivers.group_by("PARENT_DEALER_CODE_MODEL_FAMILY").agg(
    # How many months did this series qualify as a top driver?
    F.count("MONTH").alias("MONTHS_AS_TOP_DRIVER"),
    
    # What is their average sales volume when they are in the top tier?
    F.round(F.avg("SERIES_MONTHLY_SALES"), 2).alias("AVG_MONTHLY_SALES")
)

# 3. Sort by highest frequency and highest average sales
# (This brings the most consistently dominant series to the top)
df_consistency.sort(F.desc("MONTHS_AS_TOP_DRIVER"), F.desc("AVG_MONTHLY_SALES")).show()

-----------------------------------------------------------------------------------------------------
|"PARENT_DEALER_CODE_MODEL_FAMILY"                   |"MONTHS_AS_TOP_DRIVER"  |"AVG_MONTHLY_SALES"  |
-----------------------------------------------------------------------------------------------------
|10421<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK           |42                      |457.88               |
|10072<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK           |42                      |437.93               |
|10671<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK           |42                      |405.50               |
|10267<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK AND A...  |42                      |397.64               |
|10075<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK           |42                      |396.69               |
|10027<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK RED P...  |42                      |396.50               |
|10258<>SPLENDOR+<>DRUM<>SELF<>CAST<>BLACK           |42                      |390

In [26]:
df_consistency.select(F.min("MONTHS_AS_TOP_DRIVER").alias("MIN_MONTHS_AS_TOP_DRIVER"),F.max("MONTHS_AS_TOP_DRIVER").alias("MAX_MONTHS_AS_TOP_DRIVER"),F.count_distinct("PARENT_DEALER_CODE_MODEL_FAMILY").alias("UNIQUE_VALUES_IN_PARENT_DEALER_CODE_MODEL_FAMILY_COLUMN")).show()

----------------------------------------------------------------------------------------------------------------
|"MIN_MONTHS_AS_TOP_DRIVER"  |"MAX_MONTHS_AS_TOP_DRIVER"  |"UNIQUE_VALUES_IN_PARENT_DEALER_CODE_MODEL_FAMI...  |
----------------------------------------------------------------------------------------------------------------
|1                           |42                          |31895                                               |
----------------------------------------------------------------------------------------------------------------



In [27]:
#Series who were in top 85% driver for at least 3 months
pandas_form=df_consistency.filter(F.col("MONTHS_AS_TOP_DRIVER")>=1).select(F.col("PARENT_DEALER_CODE_MODEL_FAMILY")).distinct().to_pandas()

In [28]:
len(pandas_form)

31893

### Calculating the sales as per a list of series

In [29]:
# #Query to get the actual sales
with open(r"C:\Users\G0004878\Desktop\TFT_Data\Sep_forecast_Oct_SOQ\Step 1 - Data Preparation\query_to_obtain_customer_retail_sales.txt",encoding='utf-8') as f:
    sql_query = f.read()

actual_sales_df = session.sql(sql_query)

#Filter the actual sales 
# actual_sales_filtered_data = actual_sales_df.filter(
#     F.col("CAL_DATE").between(F.lit('2023-09-01'), F.lit('2023-12-07')) |
#     F.col("CAL_DATE").between(F.lit('2024-09-01'), F.lit('2024-12-07')) |
#     F.col("CAL_DATE").between(F.lit('2025-09-01'), F.lit('2025-12-07'))
# )


actual_sales_filtered_data = actual_sales_df.with_column("YEAR",F.year("CAL_DATE"))

#Aggregate by cal_date 
actual_sales_df_agg = actual_sales_filtered_data.group_by("YEAR").agg(F.sum(F.col("ECR")).alias("TOTAL_ACTUAL_DAILY_SALES")).sort(F.col("YEAR").asc())

actual_sales_df_agg.show()

---------------------------------------
|"YEAR"  |"TOTAL_ACTUAL_DAILY_SALES"  |
---------------------------------------
|2022    |3878206.000000              |
|2023    |5356798.000000              |
|2024    |5408233.000000              |
|2025    |5675046.000000              |
|2026    |4108620.000000              |
---------------------------------------



In [30]:
list_of_series=pandas_form.iloc[:,0].unique().tolist()

In [31]:
def return_sales_comparison(df,actual_sales_df_agg,list_of_series):
    #Calculate the sales for the selected series 
    new_df = df.filter(F.col("PARENT_DEALER_CODE_MODEL_FAMILY").isin(list_of_series))

    new_df_agg = new_df.group_by(F.year("CAL_DATE").alias("YEAR")).agg(F.sum("NET_SALES").alias("TOTAL_YEARLY_SALES_FOR_SELECTED_SERIES")).sort(F.col("YEAR").asc())

    joined_df = new_df_agg.join(actual_sales_df_agg,on="YEAR")

    joined_df = joined_df.with_column("PERCENTAGE_OF_SALES_CAPTURED",F.round(F.col("TOTAL_YEARLY_SALES_FOR_SELECTED_SERIES")/F.col("TOTAL_ACTUAL_DAILY_SALES"),4)*100)

    return joined_df
    


In [32]:
(65.99+89.46+89.15+85.26)/4

82.465

In [33]:
comparison_df=return_sales_comparison(df,actual_sales_df_agg,list_of_series)

comparison_df.show()

-------------------------------------------------------------------------------------------------------------------
|"YEAR"  |"TOTAL_YEARLY_SALES_FOR_SELECTED_SERIES"  |"TOTAL_ACTUAL_DAILY_SALES"  |"PERCENTAGE_OF_SALES_CAPTURED"  |
-------------------------------------------------------------------------------------------------------------------
|2023    |3707100.000000                            |5356798.000000              |69.2000                         |
|2024    |5027370.000000                            |5408233.000000              |92.9600                         |
|2025    |5257180.000000                            |5675046.000000              |92.6400                         |
|2026    |3642561.000000                            |4108620.000000              |88.6600                         |
-------------------------------------------------------------------------------------------------------------------



In [34]:
comparison_df.select(F.avg("PERCENTAGE_OF_SALES_CAPTURED")).show()

-------------------------------------------
|"AVG(""PERCENTAGE_OF_SALES_CAPTURED"")"  |
-------------------------------------------
|85.8650000000                            |
-------------------------------------------



In [35]:
#Valid series
valid_series_snowpark_dataframe = session.create_dataframe(pandas_form)

In [36]:
valid_series_snowpark_dataframe.write.mode('overwrite').save_as_table("MOP_DATABASE.SOQ.VALID_SERIES_FOR_DAILY_MODELLING")

In [37]:
data = remove_quotes_existing_columns(data)

In [38]:
all_series = set(data.select(F.col("PARENT_DEALER_CODE_MODEL_FAMILY")).distinct().to_pandas().iloc[:,0].unique().tolist())

In [39]:
valid_series = set(list_of_series)

In [40]:
invalid_series = all_series - valid_series

print(f"Length of all_series is {len(list(all_series))}")

print(f"Length of valid_series is {len(list(valid_series))}")

print(f"Length of invalid_series is {len(list(invalid_series))}")

print(f"Total length is : {len(list(valid_series)) + len(list(invalid_series))}")

Length of all_series is 104087
Length of valid_series is 31893
Length of invalid_series is 72194
Total length is : 104087


In [41]:
invalid_series_snowpark_df = session.create_dataframe(data=list(invalid_series),schema=["INVALID_SERIES"])

In [42]:
invalid_series_snowpark_df.show()

------------------------------------------------------
|"INVALID_SERIES"                                    |
------------------------------------------------------
|10019<>GLAMOUR<>DISC<>SELF<>CAST<>BLUE              |
|10104<>DESTINI<>DRUM<>SELF<>SHEET METAL<>RED        |
|11075<>XOOM<>DISC<>SELF<>CAST<>GREY                 |
|12047<>SPLENDOR+<>DRUM<>SELF<>CAST<>GOLD            |
|10869<>GLAMOUR<>DRUM<>SELF<>CAST<>BLACK HEAVY GREY  |
|10737<>PASSION<>DRUM<>SELF<>CAST<>BLACK             |
|10045<>HF DELUXE<>DRUM<>KICK<>CAST<>BLUE            |
|12162<>PLEASURE+<>DRUM<>SELF<>CAST<>BLACK           |
|10940<>GLAMOUR<>DRUM<>SELF<>CAST<>RED BLACK         |
|11845<>XOOM<>DISC<>SELF<>CAST<>BLACK                |
------------------------------------------------------



In [43]:
invalid_series_snowpark_df.write.mode('overwrite').save_as_table("MOP_DATABASE.SOQ.INVALID_SERIES_FOR_DAILY_MODELLING")